In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import warnings
warnings.filterwarnings('ignore')

In [2]:
df = sns.load_dataset('tips')
df.head()

,total_bill,tip,sex,smoker,day,time,size
0,16.99,1.01,Female,No,Sun,Dinner,2
1,10.34,1.66,Male,No,Sun,Dinner,3
2,21.01,3.50,Male,No,Sun,Dinner,3
3,23.68,3.31,Male,No,Sun,Dinner,2
4,24.59,3.61,Female,No,Sun,Dinner,4


In [3]:
from sklearn.ensemble import RandomForestClassifier
rf = RandomForestClassifier()
rf

RandomForestClassifier()

In [5]:
df['time'].unique()

['Dinner', 'Lunch']
Categories (2, object): ['Lunch', 'Dinner']

In [7]:
df.isnull().sum()

,0
total_bill,0
tip,0
sex,0
smoker,0
day,0
time,0
size,0


In [8]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 244 entries, 0 to 243
Data columns (total 7 columns):
 #   Column      Non-Null Count  Dtype   
---  ------      --------------  -----   
 0   total_bill  244 non-null    float64 
 1   tip         244 non-null    float64 
 2   sex         244 non-null    category
 3   smoker      244 non-null    category
 4   day         244 non-null    category
 5   time        244 non-null    category
 6   size        244 non-null    int64   
dtypes: category(4), float64(2), int64(1)
memory usage: 7.4 KB


In [9]:
from sklearn.preprocessing import LabelEncoder
encoder = LabelEncoder()
df['time'] = encoder.fit_transform(df['time'])
df

,total_bill,tip,sex,smoker,day,time,size
0,16.99,1.01,Female,No,Sun,0,2
1,10.34,1.66,Male,No,Sun,0,3
2,21.01,3.50,Male,No,Sun,0,3
3,23.68,3.31,Male,No,Sun,0,2
4,24.59,3.61,Female,No,Sun,0,4
...,...,...,...,...,...,...,...
239,29.03,5.92,Male,No,Sat,0,3
240,27.18,2.00,Female,Yes,Sat,0,2
241,22.67,2.00,Male,Yes,Sat,0,2
242,17.82,1.75,Male,No,Sat,0,2


In [10]:
df['time'].unique()

array([0, 1])

In [12]:
X = df.drop('time', axis= 1)
y = df['time']

In [14]:
X.shape, y.shape

((244, 6), (244,))

In [15]:
X

,total_bill,tip,sex,smoker,day,size
0,16.99,1.01,Female,No,Sun,2
1,10.34,1.66,Male,No,Sun,3
2,21.01,3.50,Male,No,Sun,3
3,23.68,3.31,Male,No,Sun,2
4,24.59,3.61,Female,No,Sun,4
...,...,...,...,...,...,...
239,29.03,5.92,Male,No,Sat,3
240,27.18,2.00,Female,Yes,Sat,2
241,22.67,2.00,Male,Yes,Sat,2
242,17.82,1.75,Male,No,Sat,2


In [16]:
y

,time
0,0
1,0
2,0
3,0
4,0
...,...
239,0
240,0
241,0
242,0


In [17]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size= 0.2)
X_train.shape, X_test.shape, y_train.shape, y_test.shape

((195, 6), (49, 6), (195,), (49,))

In [18]:
X_train.head()

,total_bill,tip,sex,smoker,day,size
63,18.29,3.76,Male,Yes,Sat,4
201,12.74,2.01,Female,Yes,Thur,2
209,12.76,2.23,Female,Yes,Sat,2
127,14.52,2.00,Female,No,Thur,2
32,15.06,3.00,Female,No,Sat,2


In [19]:
X_test.head()

,total_bill,tip,sex,smoker,day,size
160,21.50,3.5,Male,No,Sun,4
74,14.73,2.2,Female,No,Sat,2
212,48.33,9.0,Male,No,Sat,4
223,15.98,3.0,Female,No,Fri,3
83,32.68,5.0,Male,Yes,Thur,2


In [21]:
y_train

,time
63,0
201,1
209,0
127,1
32,0
...,...
70,0
88,1
166,0
211,0


In [22]:
y_test

,time
160,0
74,0
212,0
223,1
83,1
168,0
66,0
13,0
21,0
220,1


In [23]:
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

In [24]:
df.sample(2)

,total_bill,tip,sex,smoker,day,time,size
110,14.00,3.00,Male,No,Sat,0,2
201,12.74,2.01,Female,Yes,Thur,1,2


In [25]:
cat_cols = ["sex", "smoker", "day"]
num_cols = ["total_bill", "tip", "size"]

In [26]:
cat_pipeline = Pipeline(steps= [("imputation", SimpleImputer(strategy= "most_frequent")),
                                ("encoding", OneHotEncoder())])
num_pipeline = Pipeline(steps= [("imputation", SimpleImputer(strategy= "mean")),
                                ("scaling", StandardScaler())])

In [27]:
preprocessor = ColumnTransformer([("num_pipeline", num_pipeline, num_cols),
                                  ("cat_pipeline", cat_pipeline, cat_cols)])
preprocessor

ColumnTransformer(transformers=[('num_pipeline',
                                 Pipeline(steps=[('imputation',
                                                  SimpleImputer()),
                                                 ('scaling',
                                                  StandardScaler())]),
                                 ['total_bill', 'tip', 'size']),
                                ('cat_pipeline',
                                 Pipeline(steps=[('imputation',
                                                  SimpleImputer(strategy='most_frequent')),
                                                 ('encoding',
                                                  OneHotEncoder())]),
                                 ['sex', 'smoker', 'day'])])

In [28]:
X_train = preprocessor.fit_transform(X_train)
X_test = preprocessor.transform(X_test)

In [29]:
X_train

array([[-0.10446322,  0.63211664,  1.51092145, ...,  1.        ,
         0.        ,  0.        ],
       [-0.76054102, -0.73990857, -0.55665527, ...,  0.        ,
         0.        ,  1.        ],
       [-0.75817677, -0.5674254 , -0.55665527, ...,  1.        ,
         0.        ,  0.        ],
       ...,
       [ 0.18752095, -0.55958526, -0.55665527, ...,  0.        ,
         1.        ,  0.        ],
       [ 0.79394961,  1.72973681,  1.51092145, ...,  1.        ,
         0.        ,  0.        ],
       [ 3.42771776,  1.6042945 ,  3.57849818, ...,  0.        ,
         1.        ,  0.        ]])

In [30]:
X_test

array([[ 0.27499798,  0.42827289,  1.51092145,  0.        ,  1.        ,
         1.        ,  0.        ,  0.        ,  0.        ,  1.        ,
         0.        ],
       [-0.52529871, -0.59094583, -0.55665527,  1.        ,  0.        ,
         1.        ,  0.        ,  0.        ,  1.        ,  0.        ,
         0.        ],
       [ 3.44663171,  4.74035212,  1.51092145,  0.        ,  1.        ,
         1.        ,  0.        ,  0.        ,  1.        ,  0.        ,
         0.        ],
       [-0.37753344,  0.03626569,  0.47713309,  1.        ,  0.        ,
         1.        ,  0.        ,  1.        ,  0.        ,  0.        ,
         0.        ],
       [ 1.59661055,  1.6042945 , -0.55665527,  0.        ,  1.        ,
         0.        ,  1.        ,  0.        ,  0.        ,  0.        ,
         1.        ],
       [-1.01469728, -1.05351433, -0.55665527,  1.        ,  0.        ,
         0.        ,  1.        ,  0.        ,  1.        ,  0.        ,
         0.   

In [32]:
from sklearn.ensemble import RandomForestClassifier
rf_clf = RandomForestClassifier()
rf_clf

RandomForestClassifier()

In [34]:
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV

params = {
    'max_depth': [1,2,3,4,5,10,None],
    'n_estimators': [30, 40, 50, 100, 200, 300],
    'criterion': ['gini', 'entropy']
}

params

{'max_depth': [1, 2, 3, 4, 5, 10, None],
 'n_estimators': [30, 40, 50, 100, 200, 300],
 'criterion': ['gini', 'entropy']}

In [36]:
clf = RandomizedSearchCV(rf_clf, param_distributions= params, cv= 5, verbose= 3, scoring= 'accuracy', n_iter= 10)
clf

RandomizedSearchCV(cv=5, estimator=RandomForestClassifier(),
                   param_distributions={'criterion': ['gini', 'entropy'],
                                        'max_depth': [1, 2, 3, 4, 5, 10, None],
                                        'n_estimators': [30, 40, 50, 100, 200,
                                                         300]},
                   scoring='accuracy', verbose=3)

In [38]:
clf.fit(X_train, y_train)

Fitting 5 folds for each of 10 candidates, totalling 50 fits
[CV 1/5] END criterion=entropy, max_depth=5, n_estimators=200;, score=0.949 total time=   0.9s
[CV 2/5] END criterion=entropy, max_depth=5, n_estimators=200;, score=0.949 total time=   0.6s
[CV 3/5] END criterion=entropy, max_depth=5, n_estimators=200;, score=0.923 total time=   0.6s
[CV 4/5] END criterion=entropy, max_depth=5, n_estimators=200;, score=0.974 total time=   0.8s
[CV 5/5] END criterion=entropy, max_depth=5, n_estimators=200;, score=1.000 total time=   0.6s
[CV 1/5] END criterion=gini, max_depth=None, n_estimators=40;, score=0.949 total time=   0.1s
[CV 2/5] END criterion=gini, max_depth=None, n_estimators=40;, score=0.949 total time=   0.1s
[CV 3/5] END criterion=gini, max_depth=None, n_estimators=40;, score=0.923 total time=   0.1s
[CV 4/5] END criterion=gini, max_depth=None, n_estimators=40;, score=0.974 total time=   0.3s
[CV 5/5] END criterion=gini, max_depth=None, n_estimators=40;, score=1.000 total time=  

RandomizedSearchCV(cv=5, estimator=RandomForestClassifier(),
                   param_distributions={'criterion': ['gini', 'entropy'],
                                        'max_depth': [1, 2, 3, 4, 5, 10, None],
                                        'n_estimators': [30, 40, 50, 100, 200,
                                                         300]},
                   scoring='accuracy', verbose=3)

In [39]:
clf.best_score_

np.float64(0.9743589743589745)

In [40]:
clf.best_params_

{'n_estimators': 50, 'max_depth': 3, 'criterion': 'entropy'}

In [41]:
final_model = clf.best_estimator_
y_pred = final_model.predict(X_test)

In [42]:
from sklearn.metrics import accuracy_score
print(accuracy_score(y_test, y_pred))

0.9387755102040817


In [43]:
# Automating the model training, evaluation

In [44]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

In [48]:
models = {
    "decision_tree": DecisionTreeClassifier(),
    "support_vector": SVC(),
    "Logistic_Reg.": LogisticRegression(),
    "Random_forest": RandomForestClassifier()
}

models

{'decision_tree': DecisionTreeClassifier(),
 'support_vector': SVC(),
 'Logistic_Reg.': LogisticRegression(),
 'Random_forest': RandomForestClassifier()}

In [49]:
def model_train_evaluation(X_train, X_test, y_train, y_test, models):
  evaluation = {}
  for i in range(len(models)):
    model = list(models.values())[i]
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    model_score = accuracy_score(y_test, y_pred)
    evaluation[list(models.keys())[i]] = model_score

  return evaluation

In [50]:
model_train_evaluation(X_train, X_test, y_train, y_test, models)

{'decision_tree': 0.9591836734693877,
 'support_vector': 0.9387755102040817,
 'Logistic_Reg.': 0.9387755102040817,
 'Random_forest': 0.9387755102040817}

In [ ]:
#internal homework

#use total_bill as target variable and make it a regressor problem
#automate fe, taraing, testing and also random forest regressor with MLR and SVR model